# Milestone 1 - System setup & Qdrant / llama.cpp diagnostics

Verifies the local, fully on-premise environment before any modelling: Python/PyTorch and the Apple-Silicon GPU (MPS) backend, the model weights on disk, the **Qdrant** vector database and the **llama.cpp** `llama-server` hosting **Phi-4-mini-instruct (3.8B, Q8_0 GGUF)**. Nothing here calls an external API.

In [1]:
import os, sys, json, time, warnings, pathlib
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 120)
from rag import config

In [2]:
import platform, subprocess, torch, transformers, qdrant_client, fastapi
info = {
    "python": platform.python_version(), "platform": platform.platform(),
    "cpu": subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True).stdout.strip(),
    "ram_gb": round(int(subprocess.run(["sysctl", "-n", "hw.memsize"], capture_output=True, text=True).stdout) / 2**30, 1),
    "torch": torch.__version__, "transformers": transformers.__version__,
    "qdrant_client": qdrant_client.__version__ if hasattr(qdrant_client, "__version__") else "n/a",
    "fastapi": fastapi.__version__, "device": config.device(),
}
pd.Series(info).to_frame("value")

,value
python,3.13.7
platform,macOS-26.5-arm64-arm-64bit-Mach-O
cpu,Apple M4 Max
ram_gb,48.0
torch,2.14.1
transformers,5.18.0
qdrant_client,n/a
fastapi,0.142.2
device,mps


### Model weights present locally

In [3]:
from huggingface_hub import scan_cache_dir
cache = {r.repo_id: r for r in scan_cache_dir().repos}
rows = []
for name, role in [(config.EMBED_MODEL, "embedder"), (config.RERANKER_MODEL, "reranker"), (config.NLI_MODEL, "guardrail NLI")]:
    rows.append({"model": name, "role": role, "size_gb": round(cache[name].size_on_disk / 1e9, 2) if name in cache else None})
rows.append({"model": config.LLM_GGUF.name, "role": "generator (llama.cpp)", "size_gb": round(config.LLM_GGUF.stat().st_size / 1e9, 2)})
ft = config.FINETUNED_RERANKER_DIR
rows.append({"model": ft.name, "role": "fine-tuned reranker", "size_gb": round(sum(f.stat().st_size for f in ft.rglob('*') if f.is_file()) / 1e9, 2) if ft.exists() else None})
pd.DataFrame(rows)

,model,role,size_gb
0,BAAI/bge-m3,embedder,6.85
1,BAAI/bge-reranker-large,reranker,2.26
2,MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli,guardrail NLI,0.88
3,Phi-4-mini-instruct-LoRA-Q8_0.gguf,generator (llama.cpp),4.08
4,bge-reranker-large-finetuned,fine-tuned reranker,2.26


### Qdrant vector database

In [4]:
from rag import store
print("Qdrant:", store.ensure_qdrant())
import requests
print(requests.get(f"{config.QDRANT_URL}/").json())
print("collections:", [c.name for c in store.client().get_collections().collections])

Qdrant: already running
{'title': 'qdrant - vector search engine', 'version': '1.19.1', 'commit': '6ab21cac18ebb6f4ae29102c7f8f5cc11affd5de'}
collections: ['company_portal', 'enterprise_docs']


### llama.cpp server with Phi-4-mini

In [5]:
from rag import generator
print("llama-server:", generator.ensure_llm())
print(subprocess.run([config.LLM_SERVER_BIN, "--version"], capture_output=True, text=True).stderr.strip().splitlines()[0])
print(requests.get(f"{config.LLM_URL}/v1/models").json()["data"][0]["id"])
r = generator.chat([{"role": "user", "content": "Reply with the single word: ready"}], max_tokens=5)
print("response:", r["text"], "| latency %.2fs" % r["latency_s"])

llama-server: started
0.00.000.083 I srv  llama_server: initializing ...
phi-4-mini-lora


response: Sure, I'm ready. | latency 0.10s


In [6]:
t = time.time(); x = torch.randn(4096, 4096, device=config.device()); y = (x @ x).sum().item()
print(f"GPU matmul smoke test on {config.device()}: {time.time() - t:.3f}s")
print("Environment verified: all components local, no external API keys configured.")

GPU matmul smoke test on mps: 0.120s
Environment verified: all components local, no external API keys configured.
